In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
import os
import numpy as np
from pathlib import Path

In [ ]:
results_dir = Path(os.environ["EPHEMERAL"]) / "nov2026_pyspoc_results/results/concat/"

# N = 500
# P = 188
# dataset_folder = f'terekhova_N{N}_P{P}_customGenes_oxphos'
# dm_file = 'datamatrix__6configs__filtered_st85_dt25.parquet'

N = 500
P = 1000
dataset_folder = f'terekhova_N{N}_P{P}_topK'
dm_file = 'datamatrix__5configs__filtered_st75_dt25.parquet'

data_path = results_dir / dataset_folder / dm_file

In [ ]:
data_matrix = pd.read_parquet(data_path)

## OPTIONAL: REMOVE PATHS FROM DATA MATRIX COLUMN NAMES
## (EXTERNAL MODULES GET SAVED WITH THE FULL PATH)

stats_path = os.path.expanduser(f'~/hcda_pyspoc_project/py-spoc/working_analyses/hjResearch_2025/summary_stats/')

data_matrix_names = data_matrix.columns.str.replace(stats_path, ".", regex=False)
data_matrix.columns = data_matrix_names

print(data_matrix.shape[0],data_matrix.shape[1])

In [ ]:
# data_matrix[data_matrix.inde`x.str.contains('blood__whole_blood__N200_P100_id0_geneid')]

In [ ]:
# data_matrix.loc[:,num_unique_val_per_stat<=1]

In [ ]:
short_stat_names = pd.Series(data_matrix.columns.str.split(".").str[-4:-1].str.join("."), index=data_matrix.columns)

In [ ]:
print('the following cols. are constant across all datasets:')
num_unique_val_per_stat = data_matrix.nunique()
print(num_unique_val_per_stat[num_unique_val_per_stat <= 1].shape)

# remove constant columns
print('removing constant cols')
df_processed = data_matrix.loc[:,~(num_unique_val_per_stat <= 1)].copy()

In [ ]:
T = df_processed.T
dups = T[T.duplicated(keep=False)] # all columns that have a copy
groups = [[len(list(g.index))] + list(g.index) for _, g in dups.groupby(list(dups.columns), dropna=False)]
groups_short = [[len(list(g.index))] + list(short_stat_names[g.index]) for _, g in dups.groupby(list(dups.columns), dropna=False)]

## ---- FASTER WAY TO FIND DUPS, HASH THE ENTIRE COL
# h = df_processed.apply(lambda c: hash(c.to_numpy().tobytes())) # one hash per column
# groups = [list(cols) for cols in h.groupby(h).groups.values() if len(cols) > 1]
# for g in groups[:10]:
#     print(g)

print(len(groups), "groups of identical columns")
df_dup_cols = pd.DataFrame(groups_short)

In [ ]:
df_dup_cols.sort_values(by=0,ascending=True).drop_duplicates()

In [ ]:
print("NaN:", np.isnan(df_processed).sum().sum(), " inf:", np.isinf(df_processed).sum().sum(), " max |x|:", np.nanmax(np.abs(df_processed)))

In [ ]:
df_dedup = df_processed.T.drop_duplicates().T.copy()

# CURRENTLY I MANUALLY MAKE THE DETERMINNANT LOG
# TO BE IMPLEMENTED IN LIBRARY
detCols = [v for v in df_dedup.columns if 'pyspoc.reducers.basic.Determinant.scaled' in v]

# clipping above at 1e200 to remove inf
logdet_per_dim = (np.log(df_dedup[detCols].abs().clip(lower=1e-300,upper=1e200))) / P
sign = np.sign(df_dedup[detCols])
df_dedup[detCols] = logdet_per_dim

In [ ]:
df_processed.shape[0] == df_processed.drop_duplicates().shape[0]

In [ ]:
df_processed.shape[1]-len(groups) == df_dedup.shape[1]

In [ ]:
data_matrix.shape, df_dedup.shape

In [ ]:
(df_dedup.max(axis=0)-df_dedup.min(axis=0)).describe()

In [ ]:
print("NaN:", np.isnan(df_dedup).sum().sum(), " inf:", np.isinf(df_dedup).sum().sum(), " max |x|:", np.nanmax(np.abs(df_dedup)))

In [ ]:
# # CHECK FOR EXTREMELY BIG VALUES:
# mx = df_dedup.max().sort_values(ascending=False)
# print(mx.head(15))
# big = df_dedup.columns[df_dedup.max() > 1e40]
# print(len(big), "columns above 1e40")
# print(df_dedup[big].idxmax().value_counts().head())

### clustering the data matrix

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

In [ ]:
nb_dir = Path("hcda_pyspoc_project/py-spoc/working_analyses/hjResearch_2025/hpc_pyspoc_scripts/py_scripts/dataviz_notebooks/")
FIG_DIR = Path(os.environ["HOME"]) / nb_dir / 'figures' / dataset_folder
os.makedirs(FIG_DIR,exist_ok=True)

In [ ]:
meta_dir = Path(os.environ["HOME"]) / "hcda_pyspoc_project/py-spoc/working_analyses/hjResearch_2025/hpc_pyspoc_scripts/metadata_files/"
meta_fl = meta_dir / f"{dataset_folder}.csv"
meta_df_all = pd.read_csv(meta_fl)
meta_df_all['donor_id'] = meta_df_all['batch_donor_id'].map(lambda v : v.split('_')[1])

meta_df = meta_df_all[['sample_id','donor_id','age']].drop_duplicates().set_index('sample_id')

In [ ]:
# sample_id = df_dedup.index.str.split('_cells').str[0]
# donor_id = df_dedup.index.map(meta_df['donor_id'].to_dict())
# age = df_dedup.index.map(meta_df['age'].to_dict())

In [ ]:
df_module_cluster = df_dedup.sample(frac=1,random_state=12).sample(frac=1, axis=1, random_state=2090)

In [ ]:
# df_module_cluster

In [ ]:
# df_dedup.columns.sample(frac=1)

### pca, tsne viz

In [ ]:
def label_age_grp(x):
    # edges = np.arange(40, 90, 5)
    edges = [25,35,45,55,65,75,85]
    labels = np.array([f"<={edges[0]}"] + [f"{a+1}-{b}" for a, b in zip(edges[:-1], edges[1:])] + [f">{edges[-1]}"])
    return labels[np.digitize(x, edges, right=True)]

In [ ]:
X = StandardScaler().fit_transform(df_module_cluster)

n_pcs = 50
pca = PCA(n_components=n_pcs)
Z = pca.fit_transform(X)

sample_id = df_module_cluster.index.str.split('_cells').str[0]
donor_id = sample_id.map(meta_df['donor_id'].to_dict())
age_map = sample_id.map(meta_df['age'].to_dict())
age_grp = age_map.map(label_age_grp)

In [ ]:
n_pcs_plot=10
xv = [v+1 for v in range(n_pcs_plot)]
plt.plot(xv,pca.explained_variance_ratio_[:n_pcs_plot],marker='o')
plt.xlabel('number of components')
plt.ylabel('cumulative explained variance')
plt.xticks(xv)

figname = f'PCA_explainedVar_{n_pcs_plot}PCs_{dataset_folder}'
plt.savefig(FIG_DIR / f"{figname}.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
n_pcs_plot=5
pc_names = [f"PC{i + 1}" for i in range(n_pcs_plot)]
Z_pl = Z[:,:n_pcs_plot]

pc_df = pd.DataFrame(Z_pl, columns=pc_names)
pc_df["coarse"] = age_grp.to_numpy()

# --- pairplot ---

g = sns.PairGrid(pc_df, x_vars=pc_names[:-1], y_vars=pc_names[1:], hue="coarse",
                 hue_order=None, palette='tab10', height=2.2, corner=True)
g.map_offdiag(sns.scatterplot, s=8, alpha=0.7, linewidth=0)
# g.add_legend()
# sns.move_legend(g, "upper right", bbox_to_anchor=(0.98, 0.98), ncol=2,
                # title=None, frameon=True, fontsize=8, markerscale=1.5)
# g.legend.set_title(None)

plt.tight_layout()
figname = f'PCA_coarse_{n_pcs_plot}PCs_{dataset_folder}'
plt.savefig(FIG_DIR / f"{figname}.png", dpi=300, bbox_inches="tight")
plt.show()

##### cluster visualizations

In [ ]:
import umap
from sklearn.manifold import TSNE

In [ ]:
X = StandardScaler().fit_transform(df_module_cluster)

E = TSNE(n_components=2, perplexity=30, init="pca",random_state=2389).fit_transform(X)
# E = umap.UMAP(metric="correlation").fit_transform(X)

In [ ]:
plot_df = pd.DataFrame({"tSNE1": E[:, 0], "tSNE2": E[:, 1], "coarse":age_grp.to_numpy(),
                        "fine":age_map.to_numpy()})

fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(data=plot_df, x="tSNE1", y="tSNE2", hue="coarse",
                hue_order=None,
                palette='tab10', s=12, alpha=0.8, linewidth=0)
ax.legend(fontsize=8, markerscale=1.5, bbox_to_anchor=(1, -0.5), loc="lower right",ncol=4)

plt.tight_layout()
figname = f'tSNE_all_coarse_{dataset_folder}'
plt.savefig(FIG_DIR / f"{figname}.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import numpy as np, math

hue_key = "coarse"
N = 1 # coarse labels highlighted per panel
cats = list(plot_df.sort_values(by='fine')[hue_key].unique())
chunks = [cats[i:i + N] for i in range(0, len(cats), N)]
ncols = 3
nrows = math.ceil(len(chunks) / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4.5 * nrows),
                         sharex=True, sharey=True, squeeze=False)

for ax, chunk in zip(axes.flat, chunks):
    
    ax.scatter(plot_df.tSNE1, plot_df.tSNE2, c="lightgrey", s=6, linewidth=0)  # background
    sub = plot_df[plot_df[hue_key].isin(chunk)]
    sns.scatterplot(data=sub, x="tSNE1", y="tSNE2", hue=hue_key, hue_order=chunk,
                    palette=sns.color_palette("tab10", len(chunk)),
                    s=12, alpha=0.8, linewidth=0, ax=ax)
    ax.legend(fontsize=7, markerscale=1.5, loc="best")

for ax in axes.flat[len(chunks):]: # hide unused panels
    ax.axis("off")

plt.tight_layout()
figname = f'tSNE_subplots_{hue_key}_{dataset_folder}'
plt.savefig(FIG_DIR / f"{figname}.png", dpi=300, bbox_inches="tight")
plt.show()

### dataset kNN's

In [ ]:
from sklearn.neighbors import NearestNeighbors

In [ ]:

def get_pca_knn_tk(k,var_exp,df_module_cluster):

    X = StandardScaler().fit_transform(df_module_cluster)

    pca = PCA(n_components=var_exp)
    Z = pca.fit_transform(X)
    n_comps = pca.n_components_
    # print(f'{n_comps} pca components')
    
    nn = NearestNeighbors(n_neighbors=k + 1).fit(Z)
    dist, idx = nn.kneighbors(Z)
    idx, dist = idx[:, 1:], dist[:, 1:]

    ids = df_module_cluster.index.to_numpy()

    sample_id = df_module_cluster.index.str.split('_cells').str[0]
    batch_id = sample_id.str.split('_').str[0]
    donor_id = sample_id.map(meta_df['donor_id'].to_dict())
    age_map = sample_id.map(meta_df['age'].to_dict())
    age_grp = age_map.map(label_age_grp)

    
    pca_knn = pd.DataFrame({
        "dataset": ids,
        "neighbors": [list(ids[i]) for i in idx],
        "nb_sample_id":[set(sample_id[i]) for i in idx],
        "nb_batch_id":[set(batch_id[i]) for i in idx],        
        "nb_donor_id":[set(donor_id[i]) for i in idx],
        "nb_age_map":[set(age_map[i]) for i in idx],   
        "nb_age_grp":[set(age_grp[i]) for i in idx],   
        "mean_dist": dist.mean(axis=1),
    })

    return pca_knn,n_comps

In [ ]:
k = 2

pca_knn_dct = {}

for k in [2,3,4,5,6,7,8,9]:
    pca_knn_dct[k] = {}
    
    for var_exp in [0.7,0.75,0.8,0.85,0.9]:
        pc_df, nc = get_pca_knn_tk(k,var_exp,df_module_cluster)        
        pca_knn_dct[k][(nc,var_exp)] = pc_df


In [ ]:
def get_nb_agreement_tk(pca_knn):
    vc_1_prop_nb_agree = {}
    for sv in ['sample_id','donor_id','age_map','age_grp','batch_id']:
        pca_knn[f'{sv}_nunique'] = pca_knn[f'nb_{sv}_set'].apply(lambda v: len(v))
        vc_1_prop_nb_agree[f'{sv}_nunique'] = pca_knn[f'{sv}_nunique'].value_counts()[1] / pca_knn.shape[0]
    return vc_1_prop_nb_agree

def get_i_agrement_tk(pca_knn,permut_mode=False):
    
    vc_1_prop = {}

    ids = pca_knn["dataset"]
    pos = {d: i for i, d in enumerate(ids)}
    idx = np.array([[pos[n] for n in nbs] for nbs in pca_knn["neighbors"]])

    sample_id = pca_knn['dataset'].str.split('_cells').str[0]
    batch_id = sample_id.str.split('_').str[0]    
    donor_id = sample_id.map(meta_df['donor_id'].to_dict())
    age_map = sample_id.map(meta_df['age'].to_dict())
    age_grp = age_map.map(label_age_grp)
    
    check_same = lambda lab : (lab[idx] == lab[:, None]).all(axis=1)

    sv_labs_lst = [("sample_id", sample_id), ("donor_id", donor_id),
                  ("age_map",age_map),("age_grp",age_grp),("batch_id",batch_id)]
    
    if permut_mode == False:    
        for sv, lab in sv_labs_lst:        
            lab = np.asarray(lab)
            all_same = check_same(lab) # Turue if every neighbor has the dataset's own label        
            vc_1_prop[sv] = all_same.mean()
            
        return vc_1_prop

    else:
        seedval=32420904238
        n_permut = 20
        rng = np.random.default_rng(seedval)
        
        for sv, lab in sv_labs_lst:        
            all_s_lst = []
            for j in range(n_permut):
                lab_perm = rng.permutation(lab)
                all_same = check_same(lab_perm)
                all_s_lst.append(all_same.mean())
                
            vc_1_prop[sv] = np.array(all_s_lst).mean()
            
        return vc_1_prop    

In [ ]:
vc_props_dct = {}
permut_props_dct = {}

for k1 in pca_knn_dct.keys():
    
    vc_props_dct[k1] = {}
    permut_props_dct[k1] = {}
    
    for k2 in pca_knn_dct[k1].keys():        
        # print(k1,k2)
        vc_props_dct[k1][k2] = get_i_agrement_tk(pca_knn_dct[k1][k2],permut_mode=False)

        permut_props_dct[k1][k2] = get_i_agrement_tk(pca_knn_dct[k1][k2],permut_mode=True)

In [ ]:
outs_dct = {"permuted": permut_props_dct, "true": vc_props_dct}

In [ ]:
rows = [(type_v, k, var, q, v) for type_v,dct_v in outs_dct.items()
                        for k, d1 in dct_v.items()
                       for var, d2 in d1.items()
                       for q, v in d2.items()]

long = pd.DataFrame(rows, columns=["type","k NN", "pca_key", "label", "value"])
long['n_comp'] = long['pca_key'].apply(lambda v : v[0])
long['exp_var'] = long['pca_key'].apply(lambda v : v[1])

g = sns.relplot(data=long, x="k NN", y="value", hue="n_comp", col="label",row="type",
                kind="line", marker="o", height=3, facet_kws={"sharey": None})

g.figure.suptitle("Local agreement of dataset labels in PCA space", y=1.02)
figname=f'lab_agreement_{dataset_folder}'
g.savefig(FIG_DIR / f"{figname}.png", dpi=300, bbox_inches="tight")

In [ ]:
# g = sns.relplot(data=long, x="n_comp", y="value", hue="k NN", col="plot_qt",
#kind="line", marker="o", col_wrap=3, height=3, facet_kws={"sharey": False})

In [ ]:
pca_knn_dct[3][(18,0.8)].columns

In [ ]:
## look at different tissue type NN's

pca_df = pca_knn_dct[3][(18,0.8)]

label_val = 'batch_id'
pca_df[f'{label_val}_set_len'] = pca_df[f'nb_{label_val}'].apply(lambda v : len(v))
pca_overlap_fine = pca_df[pca_df[f'{label_val}_set_len']>1]
fine_set_counts = pca_overlap_fine[f'nb_{label_val}'].value_counts().sort_values(ascending=False)

In [ ]:
pca_overlap_fine.shape, pca_df.shape

In [ ]:
fine_set_counts[:10]

## hierarchical clustering

In [ ]:
from scipy.cluster.hierarchy import linkage, fcluster
from sklearn.metrics import homogeneity_completeness_v_measure as hcv

In [ ]:
X = pd.DataFrame(
    StandardScaler().fit_transform(df_module_cluster),
    index=df_module_cluster.index,
    columns=df_module_cluster.columns,
)

link_method = "ward"
link_metric = "euclidean"
fclust_crit = "maxclust"

row_Z = linkage(X.values,   method=link_method, metric=link_metric)
col_Z = linkage(X.values.T, method=link_method, metric=link_metric)

cluster_method_str = f"hierarchical clustering; linkage: {link_method}, {link_metric}; cluster criterion: {fclust_crit}"

In [ ]:
# number of clusters to colour

k_rows = 20
row_cl = fcluster(row_Z, k_rows, criterion=fclust_crit)

k_cols = 10
col_cl = fcluster(col_Z, k_cols, criterion=fclust_crit)


In [ ]:
# X.columns

In [ ]:
def to_colors(labels, index, name, pal="tab10"):
    uniq = sorted(set(labels))
    lut = dict(zip(uniq, sns.color_palette(pal, len(uniq))))
    return pd.Series(labels, index=index, name=name).map(lut)

row_colors = pd.concat([
    to_colors(row_cl,X.index, "TK. individual ds.", pal="tab20"),
], axis=1)
row_linkage_val=row_Z

col_colors = pd.concat([
    to_colors(col_cl,X.columns, "PySpoc features", pal="tab20b"),
], axis=1)


g = sns.clustermap(
    X,
    row_linkage=row_linkage_val, col_linkage=col_Z,
    row_colors=row_colors, col_colors=col_colors,
    cmap="vlag", center=0, vmin=-2, vmax=2,
    xticklabels=False, yticklabels=False,
    figsize=(12, 12),
    dendrogram_ratio=(0.15, 0.15),
    colors_ratio=0.02,
    cbar_kws={"orientation": "horizontal", "label": "z-score"},
    rasterized=True,
)

g.gs.update(bottom=0.12)
pos = g.ax_heatmap.get_position()
g.ax_cbar.set_position([pos.x0, 0.04, pos.width, 0.015])

g.figure.suptitle(f"{cluster_method_str}\n{dataset_folder}\nclustering on {X.shape[0]} datasets vs {X.shape[1]} pyspoc features",
                 y=1.05)

figname=f"dendrogram_{dataset_folder}_row{k_rows}_col{k_cols}"
plt.savefig(FIG_DIR / f"{figname}.png", dpi=300, bbox_inches="tight")